In [83]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from cvxopt import matrix, solvers

In [52]:
df = pd.read_csv('dataset/GENDER_CLASSIFICATION.csv')
df.head()

,feat_1,feat_2,feat_3,feat_4,feat_5,feat_6,feat_7,feat_8,feat_9,feat_10,...,feat_24,feat_25,feat_26,feat_27,feat_28,feat_29,feat_30,feat_31,feat_32,gt
0,-0.900846,0.102587,-0.397814,0.112796,2.588096,-0.192754,-0.968311,-0.490886,-0.872099,-0.288411,...,2.541431,1.739102,0.166066,4.584869,-0.107031,-0.913990,-0.686416,-0.368085,-0.870545,0
1,-0.838868,0.039976,-0.387101,0.055413,2.066874,-0.226948,-0.947416,-0.472817,-0.855387,-0.207101,...,1.991721,1.259745,0.065058,3.019790,-0.110633,-0.890023,-0.611625,-0.298235,-0.855208,0
2,-0.814961,-0.010184,-0.397147,0.092713,1.897454,-0.269387,-0.945285,-0.449579,-0.849705,-0.151179,...,1.822978,1.105511,0.065353,2.500681,-0.052730,-0.885691,-0.583346,-0.218140,-0.856456,0
3,-0.110470,0.027849,-0.044310,-0.005343,0.177831,-0.232092,-0.562700,-0.400713,-0.552356,0.037349,...,-0.098367,-0.370318,-0.123008,-0.861314,0.106840,-0.483669,-0.224164,0.147321,-0.615051,0
4,-0.626313,-0.091985,-0.373756,-0.005083,1.172486,-0.314868,-0.885046,-0.412587,-0.818729,-0.012022,...,1.030348,0.421886,-0.068029,0.258984,-0.057158,-0.834079,-0.441066,-0.099874,-0.829539,0


In [92]:
class SVM:
    def __init__(self, C=1.0, kernel='Gaussian', gamma=None, p=3,
                 tol=1e-6):
        # hyperparams
        self.C = float(C)
        self.kernel_type = kernel 
        self.gamma = gamma    
        self.p = p
        self.tol = tol

        # learned attributes (popolati dopo fit)
        self.alphas = None        # all alpha (length n)
        self.support_ = None      # indices of support vectors
        self.X_sv = None
        self.y_sv = None
        self.alpha_sv = None      # alphas corresponding to sv
        self.b = 0.0

        # preprocessing
        self.scaler = None
        # mapping labels -> {-1, +1}
        self.label_map = None
        self.inv_label_map = None



    def kernel(self, X1, X2=None):
            if X2 is None:
                X2 = X1

            if self.kernel_type == "Gaussian":
                # Compute squared norms
                X1_sq = np.sum(X1**2, axis=1).reshape(-1, 1)
                X2_sq = np.sum(X2**2, axis=1).reshape(1, -1)
                # Pairwise squared distance
                dist_sq = X1_sq + X2_sq - 2 * X1 @ X2.T
                # RBF kernel
                K = np.exp(-self.gamma * dist_sq)
                return K

            elif self.kernel_type == "Polynomial":
                K = (X1 @ X2.T + self.gamma)**self.p
                return K
            

    def encode_labels(self, y):
        uniq = np.unique(y)
        if uniq.shape[0] != 2:
            raise ValueError("This implementation supports only binary classification.")
        # map unique[0] -> -1, unique[1] -> +1
        self.label_map = {uniq[0]: -1, uniq[1]: +1}
        self.inv_label_map = {-1: uniq[0], +1: uniq[1]}
        y_mapped = np.array([self.label_map[yy] for yy in y], dtype=float)
        return y_mapped
    
    
    def fit(self, X, y):
        """
        X: (n_samples, n_features) numpy array
        y: array-like labels (two classes)
        """
        X = np.asarray(X, dtype=float)
        y_in = np.asarray(y).ravel()

        # encode labels to -1 / +1
        y = self.encode_labels(y_in)

        # scaling
        self.scaler = StandardScaler().fit(X)
        Xs = self.scaler.transform(X)

        n_samples = Xs.shape[0]

        # compute full kernel matrix
        K = self.kernel(Xs)    # (n_samples, n_samples)

        # Build Q = (y y^T) * K
        Y = y.reshape(-1, 1)
        Q = (Y @ Y.T) * K  # elementwise multiply
        # numerical stabilizer: symmetrize & add tiny jitter
        # Q = 0.5 * (Q + Q.T) + 1e-12 * np.eye(n_samples)

        P = matrix(Q)                         # cvxopt matrix
        q = matrix(-np.ones((n_samples, 1)))  # minimize 1/2 a^T P a + q^T a -> q = -1

        # G and h for 0 <= alpha <= C
        G_std = np.vstack((-np.eye(n_samples), np.eye(n_samples)))
        h_std = np.hstack((np.zeros(n_samples), np.ones(n_samples) * self.C))
        G = matrix(G_std)
        h = matrix(h_std)

        # equality A y = 0  (sum alpha_i y_i = 0)
        A = matrix(y.reshape(1, -1))
        b = matrix(0.0)

        sol = solvers.qp(P, q, G, h, A, b)

        # extract alphas (as numpy array)
        alphas_all = np.array(sol['x']).reshape(-1)

                # keep full alphas array for diagnostics
        self.alphas = alphas_all.copy()

        # support vectors: alpha > tol
        sv_mask = alphas_all > self.tol
        sv_indices = np.where(sv_mask)[0]
        self.support_ = sv_indices
        self.X_sv = Xs[sv_mask]
        self.y_sv = y[sv_mask]
        self.alpha_sv = alphas_all[sv_mask]

        # compute bias b: average over alphas with 0 < alpha < C (free SV)
        free_sv_mask = (alphas_all > self.tol) & (alphas_all < self.C - self.tol)
        free_indices = np.where(free_sv_mask)[0]

        if free_indices.size > 0:
            b_vals = []
            for i in free_indices:
                # f(x_i) = sum_j alpha_j y_j K(x_j, x_i)
                f_i = np.sum(alphas_all * y * K[:, i])
                b_vals.append(y[i] - f_i)
            self.b = np.mean(b_vals)
        else:
            # fallback: average over all support vectors
            b_vals = []
            for idx in sv_indices:
                f_i = np.sum(alphas_all * y * K[:, idx])
                b_vals.append(y[idx] - f_i)
            self.b = np.mean(b_vals)

        # keep training-data-size alphas if needed (self.alphas already)
        return self
    

    def decision_function(self, X):
        """Return raw decision scores (float), using only SV."""
        X = np.asarray(X, dtype=float)
        Xs = self.scaler.transform(X)
        K = self.kernel(Xs, self.X_sv)  # shape (m, n_sv)
        # weights = alpha_sv * y_sv (elementwise)
        weights = self.alpha_sv * self.y_sv
        scores = K @ weights + self.b    # shape (m,)
        return np.asarray(scores).reshape(-1)

    def predict(self, X):
        scores = self.decision_function(X)
        preds = np.sign(scores)
        # map back to original labels
        return np.array([self.inv_label_map[int(p)] for p in preds])
    
    def score(self, X, y):
        y = np.asarray(y)
        y_pred = self.predict(X)
        return np.mean(y_pred == y)
    

    def get_support(self):
        return self.support_

In [85]:
df.shape

(1000, 33)

In [93]:
df_train = df.head(int(df.shape[0]*0.8))

df_test = df.tail(int(df.shape[0]*0.2))

# Split features and target
X = df_train.drop("gt", axis=1).values
y = df_train["gt"].values.reshape(-1, 1)

X_test = df_test.drop("gt", axis=1).values
y_test = df_test["gt"].values.reshape(-1, 1)

svm = SVM(C=10, gamma=0.3)

svm.fit(X, y)

     pcost       dcost       gap    pres   dres
 0: -4.3559e+02 -2.5942e+04  5e+04  3e-01  1e-14
 1: -5.4532e+02 -4.6577e+03  5e+03  1e-02  1e-14
 2: -8.8590e+02 -2.2447e+03  1e+03  4e-03  1e-14
 3: -1.0374e+03 -1.5420e+03  5e+02  1e-03  1e-14
 4: -1.0992e+03 -1.3223e+03  2e+02  2e-04  2e-14
 5: -1.1256e+03 -1.1870e+03  6e+01  2e-05  2e-14
 6: -1.1334e+03 -1.1557e+03  2e+01  1e-06  2e-14
 7: -1.1363e+03 -1.1430e+03  7e+00  1e-07  2e-14
 8: -1.1372e+03 -1.1403e+03  3e+00  2e-08  2e-14
 9: -1.1377e+03 -1.1390e+03  1e+00  2e-13  2e-14
10: -1.1380e+03 -1.1383e+03  3e-01  3e-13  2e-14
11: -1.1380e+03 -1.1382e+03  2e-01  1e-13  2e-14
12: -1.1381e+03 -1.1381e+03  2e-02  2e-13  2e-14
13: -1.1381e+03 -1.1381e+03  1e-03  3e-13  2e-14
14: -1.1381e+03 -1.1381e+03  3e-05  2e-13  2e-14
Optimal solution found.


In [ ]:
def grid_search_cv(X, y, C_grid, gamma_grid=None, degree_grid=None,
                   k=5, random_state=0):
    """
    returns: best_params dict and best_score (accuracy)
    NOTE: gamma_grid can be None (then gamma kept default), same for degree_grid.
    """
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=random_state)
    best_score = -np.inf
    best_params = None

    for C in C_grid:
        for kernel in ['Gaussian', 'Polynomial']:
            if kernel == 'Gaussian':
                gamma_iter = gamma_grid if gamma_grid is not None else [None]
                for gamma in gamma_iter:
                    scores = []
                    for train_idx, val_idx in skf.split(X, y):
                        model = SVM(C=C, kernel='Gaussian', gamma=gamma)
                        model.fit(X[train_idx], y[train_idx])
                        scores.append(model.score(X[val_idx], y[val_idx]))
                    mean_score = np.mean(scores)
                    if mean_score > best_score:
                        best_score = mean_score
                        best_params = {'C': C, 'gamma': gamma, 'kernel': 'Gaussian'}
            elif kernel == 'Polynomial':
                degree_iter = degree_grid if degree_grid is not None else [3]
                for degree in degree_iter:
                    scores = []
                    for train_idx, val_idx in skf.split(X, y):
                        model = SVM(C=C, kernel='Polynomial', p=degree)
                        model.fit(X[train_idx], y[train_idx])
                        scores.append(model.score(X[val_idx], y[val_idx]))
                    mean_score = np.mean(scores)
                    if mean_score > best_score:
                        best_score = mean_score
                        best_params = {'C': C, 'degree': degree, 'kernel': 'Polynomial'}
            else:
                raise ValueError("Unsupported kernel for grid search.")
    return best_params, best_score

In [94]:
y_pred = svm.predict(X_test)

In [95]:
print(classification_report(y_test, y_pred, zero_division=0))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00         0
           1       1.00      0.89      0.94       200

    accuracy                           0.89       200
   macro avg       0.50      0.45      0.47       200
weighted avg       1.00      0.89      0.94       200

